# Profiling ANP - Entendendo o dataset e suas propriedades

In [28]:
import polars as pl

def carregar_particao(caminho: str) -> pl.DataFrame:
    df = pl.read_csv(
    caminho,
    separator=";"
    )
    return df

def verificar_chave_candidata(df: pl.DataFrame) -> pl.DataFrame:
    duplicados = (
        df.select(
            "CNPJ da Revenda",
            "Produto",
            "Data da Coleta"
        )
        .group_by([
            "CNPJ da Revenda",
            "Produto",
            "Data da Coleta"
        ])
        .len()
        .filter(
            pl.col("len") > 1
        )
    )
    return duplicados

def analisar_particao(df: pl.DataFrame) -> dict:
    resultado = {        
        "total_linhas": df.height,
        "total_colunas": df.width,        
        "colunas": df.columns,
        "data_minima": df["Data da Coleta"].str.to_date(format="%d/%m/%Y").min(),
        "data_maxima": df["Data da Coleta"].str.to_date(format="%d/%m/%Y").max(),
        "produtos": df["Produto"].unique().sort().to_list(),
        "unidades_de_medida": df["Unidade de Medida"].unique().sort().to_list(),
        "duplicidades": verificar_chave_candidata(df).height,
        "nulos": quantidade_nulos(df),
    }
    return resultado

def quantidade_nulos(df: pl.DataFrame) -> dict:
    nulos = df.null_count().to_dicts()[0]
    nulos_com_valores = {col: count for col, count in nulos.items() if count > 0}
    return nulos_com_valores

In [29]:
df_2023_1 = carregar_particao(
    "../data/bronze/anp/automotivos/ano=2023/semestre=1/AUTOMOTIVOS_2023.01.csv"
)

df_2023_2 = carregar_particao(
    "../data/bronze/anp/automotivos/ano=2023/semestre=2/AUTOMOTIVOS_2023.02.csv"
)

resultado_2023_1 = analisar_particao(df_2023_1)
resultado_2023_2 = analisar_particao(df_2023_2)

print("Resultado 2023.1:", resultado_2023_1)
print("Resultado 2023.2:", resultado_2023_2)


Resultado 2023.1: {'total_linhas': 431576, 'total_colunas': 16, 'colunas': ['Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra', 'Unidade de Medida', 'Bandeira'], 'data_minima': datetime.date(2023, 1, 2), 'data_maxima': datetime.date(2023, 6, 30), 'produtos': ['DIESEL', 'DIESEL S10', 'ETANOL', 'GASOLINA', 'GASOLINA ADITIVADA', 'GNV'], 'unidades_de_medida': ['R$ / litro', 'R$ / m³'], 'duplicidades': 0, 'nulos': {'Numero Rua': 107, 'Complemento': 333000, 'Bairro': 828, 'Valor de Compra': 431576}}
Resultado 2023.2: {'total_linhas': 472424, 'total_colunas': 16, 'colunas': ['Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra', 'Unidade de Medida', 'Bandeira'], 'data_minima': datetime.date(20